In [1]:
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class Appointment:
    id: str  # Unique identifier
    duration: int  # Duration in 15-minute blocks (e.g., 30min = 2 blocks)

    # For existing appointments only
    day: Optional[int] = None  # 0 for first day, 1 for second day, etc.
    start_block: Optional[int] = None  # 0-31 for blocks within the day

    # Optional metadata
    title: Optional[str] = None
    client_name: Optional[str] = None

In [2]:
# Existing appointments (already scheduled)
existing_appointments = [
    Appointment(
        id="existing_1",
        duration=4,  # 1 hour (4 × 15min blocks)
        day=0,       # First day
        start_block=8,  # 11:00 AM (8 × 15min after 9:00 AM)
        title="Client Meeting",
        client_name="John Doe"
    ),
    Appointment(
        id="existing_2",
        duration=6,  # 1.5 hours
        day=1,       # Second day
        start_block=16, # 1:00 PM
        title="Project Review"
    )
]

# New appointments (need to be scheduled)
new_appointments = [
    Appointment(
        id="new_1",
        duration=2,  # 30 minutes
        title="Quick Consultation"
    ),
    Appointment(
        id="new_2",
        duration=8,  # 2 hours
        title="Deep Dive Session"
    ),
    Appointment(
        id="new_3",
        duration=10, # 2.5 hours
        title="Workshop"
    )
]

In [3]:
def time_to_blocks(hour: int, minute: int) -> int:
    """Convert time to block number (0-31 for 9AM-5PM)"""
    start_hour = 9  # 9 AM
    total_minutes = (hour - start_hour) * 60 + minute
    return total_minutes // 15

def blocks_to_time(block: int) -> tuple[int, int]:
    """Convert block number back to hour and minute"""
    start_hour = 9
    total_minutes = block * 15
    hour = start_hour + total_minutes // 60
    minute = total_minutes % 60
    return hour, minute

def duration_to_blocks(hours: int, minutes: int) -> int:
    """Convert duration to number of blocks"""
    return (hours * 60 + minutes) // 15

# Usage examples
# existing_appointments = [
#     Appointment(
#         id="meeting_1",
#         duration=duration_to_blocks(1, 30),  # 1 hour 30 minutes = 6 blocks
#         day=0,
#         start_block=time_to_blocks(10, 30),  # 10:30 AM = block 6
#         title="Team Meeting"
#     )
# ]

In [4]:
from ortools.sat.python import cp_model

def create_schedule_with_existing_appointments(
    existing_appointments: List[Appointment],
    new_appointments: List[Appointment],
    num_days: int = 2,
    blocks_per_day: int = 32
) -> cp_model.CpModel:

    model = cp_model.CpModel()
    all_intervals = []
    start_vars = {}  # To access start times later

    # Add existing appointments as fixed intervals
    for existing in existing_appointments:
        if existing.day is None or existing.start_block is None:
            raise ValueError(f"Existing appointment {existing.id} missing day/start_block")

        # Convert to absolute time slot
        start_slot = existing.day * blocks_per_day + existing.start_block
        start_var = model.NewConstant(start_slot)

        interval = model.NewIntervalVar(
            start_var,
            existing.duration,
            start_var + existing.duration,
            f'existing_{existing.id}'
        )
        all_intervals.append(interval)

    # Add new appointments as variable intervals
    for new in new_appointments:
        max_start = num_days * blocks_per_day - new.duration
        start_var = model.NewIntVar(0, max_start, f'start_{new.id}')
        start_vars[new.id] = start_var  # Store for later retrieval

        interval = model.NewIntervalVar(
            start_var,
            new.duration,
            start_var + new.duration,
            f'new_{new.id}'
        )
        all_intervals.append(interval)

    # The magic constraint!
    model.AddNoOverlap(all_intervals)

    return model, start_vars

In [5]:
# After solving

model, start_vars = create_schedule_with_existing_appointments(existing_appointments, new_appointments)
solver = cp_model.CpSolver()
status = solver.Solve(model)
blocks_per_day = 32

if status == cp_model.OPTIMAL:
    for appointment in new_appointments:
        start_slot = solver.Value(start_vars[appointment.id])
        day = start_slot // blocks_per_day
        block = start_slot % blocks_per_day
        hour, minute = blocks_to_time(block)

        print(f"Appointment {appointment.id}: Day {day}, {hour:02d}:{minute:02d}")

Appointment new_1: Day 0, 14:30
Appointment new_2: Day 0, 09:00
Appointment new_3: Day 0, 12:00


In [ ]:
def create_schedule_with_day_boundaries_v3(
    existing_appointments: List[Appointment],
    new_appointments: List[Appointment],
    num_days: int = 2,
    blocks_per_day: int = 32
) -> cp_model.CpModel:

    model = cp_model.CpModel()
    all_intervals = []
    start_vars = {}

    # Add existing appointments (unchanged)
    for existing in existing_appointments:
        start_slot = existing.day * blocks_per_day + existing.start_block
        start_var = model.NewConstant(start_slot)
        interval = model.NewIntervalVar(
            start_var, existing.duration, start_var + existing.duration,
            f'existing_{existing.id}'
        )
        all_intervals.append(interval)

    # Add new appointments with day boundary constraints
    for new in new_appointments:
        max_start = num_days * blocks_per_day - new.duration
        start_var = model.NewIntVar(0, max_start, f'start_{new.id}')
        start_vars[new.id] = start_var

        # KEY CONSTRAINT: start_block + duration <= blocks_per_day
        # Where start_block = start_var % blocks_per_day
        start_block = model.NewIntVar(0, blocks_per_day - 1, f'start_block_{new.id}')
        model.AddModuloEquality(start_block, start_var, blocks_per_day)
        model.Add(start_block + new.duration <= blocks_per_day)

        interval = model.NewIntervalVar(
            start_var, new.duration, start_var + new.duration,
            f'new_{new.id}'
        )
        all_intervals.append(interval)

    model.AddNoOverlap(all_intervals)
    return model, start_vars

# Example usage
existing = [
    Appointment(id="e1", duration=4, day=0, start_block=28)  # 4PM-5PM day 0
]

new = [
    Appointment(id="n1", duration=6),  # 1.5 hours
    Appointment(id="n2", duration=2)   # 30 minutes
]

model, start_vars = create_schedule_with_day_boundaries_v3(existing, new)
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL:
    for appt in new:
        start_slot = solver.Value(start_vars[appt.id])
        day = start_slot // 32
        block = start_slot % 32
        hour, minute = blocks_to_time(block)
        print(f"{appt.id}: Day {day}, {hour:02d}:{minute:02d}")